In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# UVRegressionAndProjection
Pipeline-first driver for unified single-electrode encoding and univariate projection/reactivation analysis.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import binomtest

from membandit.plotting_style import set_plot_style
set_plot_style()

from membandit.io import Subject
from membandit._impl.multivariate import run_smoothed_time_resolved_regression_pipeline, run_univariate_reactivation_pipeline, load_univariate_trial_level, summarize_univariate_hits_misses, build_legacy_single_elec_csv_path, run_per_electrode_2sample_cluster_test


In [ ]:
# =============================
# Configuration
# =============================

RUN_ENCODING_CSV = True
RUN_PROJECTION_NPZ = True
RUN_GROUP_STATS = True

OVERWRITE_CHECKPOINTS = False
OVERWRITE_TRIAL_LEVEL = True

USE_SURROGATES = True
N_SURROGATES = 250
N_PERMUTATIONS_ENCODING = 100
N_PERMUTATIONS_CLUSTER = 1024
CLUSTER_SEED = 42

UV_EVENT_ENCODING = 'feedback_start'
UV_EVENT_PROJECTION = 'cue_on'
UV_BAND = 'hfa'
UV_FREQ_RANGE = (70., 201.)
UV_WIN_MS = 200
UV_DOWNSAMPLE_FACTOR = 5
UV_COVARIATES = ['zpm', 'preward', 'pgamble']
UV_PREDICTORS = ['zrpe']
ROIS = ['OFC', 'dlPFC', 'HPC', 'dmPFC', 'AMY', 'ACC']

min_trials = 10

base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
trial_save_dir = '/Volumes/T7 Shield/scratch/MemoryBandit/RPE_Reactivation/trial_level'

EXPORT_LEGACY_CSV = True


# =============================
# Single-electrode example configuration
# =============================
RUN_SINGLE_ELECTRODE_EXAMPLE = True
SINGLE_SUBJECT = 'MS036'  # if unavailable in your cohort, set manually to subjects[0]
SINGLE_ROI = 'HPC'
SINGLE_PREDICTOR = UV_PREDICTORS[0]
SINGLE_ELECTRODE_MODE = 'index'  # 'index' | 'label' | 'max_abs'
SINGLE_ELECTRODE_INDEX = 0
SINGLE_ELECTRODE_LABEL = None
SINGLE_USE_HITS_MINUS_MISSES = True
SINGLE_RUN_COMPUTE_IF_MISSING = True
SINGLE_SAVE_FIG = False
SINGLE_FIG_DIR = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/UVRegressionAndProjection'


In [ ]:
# =============================
# Load subject list and behavior
# =============================

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = list(subj_df.MSSMCode.values)

learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

print(f"Subjects: {len(subjects)}")
print(f"learn_df: {learn_df.shape}, combined_df: {combined_df.shape}")


In [ ]:
# =============================
# Subject availability / ROI sanity checks
# =============================

roi_counts = {roi: 0 for roi in ROIS}
subject_has_any = 0

for subj in subjects:
    elec_file = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/Day1_reref_elec_df'
    if not os.path.exists(elec_file):
        continue
    elec_df = pd.read_csv(elec_file)
    has_any = False
    for roi in ROIS:
        n_roi = int(np.sum(elec_df.salman_region == roi))
        if n_roi > 0:
            roi_counts[roi] += 1
            has_any = True
    if has_any:
        subject_has_any += 1

print(f"Subjects with >=1 ROI electrode: {subject_has_any}/{len(subjects)}")
for roi in ROIS:
    print(f"  {roi}: {roi_counts[roi]} subjects with electrodes")


In [ ]:
# =============================
# Run smoothed time-resolved encoding (legacy-compatible CSV)
# =============================

encoding_tables = {}
encoding_csv_paths = {}

if RUN_ENCODING_CSV:
    for predictor in UV_PREDICTORS:
        print("\n" + "="*70)
        print(f"Encoding predictor: {predictor}")
        print("="*70)

        formula = f"tfr ~ 1 + {predictor} + " + " + ".join(UV_COVARIATES)
        checkpoint_dir = f"{load_dir}/checkpoints_smoothed_{UV_BAND}_{UV_EVENT_ENCODING}_{predictor}"

        df_enc = run_smoothed_time_resolved_regression_pipeline(
            subjects=subjects,
            event=UV_EVENT_ENCODING,
            predictor=predictor,
            base_dir=base_dir,
            load_dir=load_dir,
            rois=ROIS,
            formula=formula,
            freq_range=UV_FREQ_RANGE,
            win_ms=UV_WIN_MS,
            downsample_factor=UV_DOWNSAMPLE_FACTOR,
            n_permutations=N_PERMUTATIONS_ENCODING,
            covariates=UV_COVARIATES,
            min_trials=min_trials,
            seed=CLUSTER_SEED,
            n_jobs_outer=1,
            checkpoint_dir=checkpoint_dir,
            overwrite_checkpoints=OVERWRITE_CHECKPOINTS,
            verbose=True,
        )

        encoding_tables[predictor] = df_enc
        print(f"Rows: {len(df_enc)}")

        if EXPORT_LEGACY_CSV and len(df_enc) > 0:
            out_csv = build_legacy_single_elec_csv_path(
                load_dir=load_dir,
                band=UV_BAND,
                event=UV_EVENT_ENCODING,
                predictor=predictor,
                suffix='smoothed_final',
            )
            df_enc.to_csv(out_csv, index=False)
            encoding_csv_paths[predictor] = out_csv
            print(f"Saved: {out_csv}")
else:
    print("RUN_ENCODING_CSV=False; skipped")


In [ ]:
# # =============================
# # Run univariate projection pipeline (NPZ)
# # =============================

# uv_results = {}

# if RUN_PROJECTION_NPZ:
#     uv_results = run_univariate_reactivation_pipeline(
#         subjects=subjects,
#         rois=ROIS,
#         predictors=UV_PREDICTORS,
#         base_dir=base_dir,
#         load_dir=load_dir,
#         event_encoding=UV_EVENT_ENCODING,
#         event_projection=UV_EVENT_PROJECTION,
#         covariates=UV_COVARIATES,
#         freq_range=UV_FREQ_RANGE,
#         downsample_factor=UV_DOWNSAMPLE_FACTOR,
#         win_ms=UV_WIN_MS,
#         min_trials=min_trials,
#         n_surrogates=N_SURROGATES,
#         use_surrogates=USE_SURROGATES,
#         seed=CLUSTER_SEED,
#         save_trial_level=True,
#         trial_save_dir=trial_save_dir,
#         overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
#         trial_save_dtype=np.float32,
#         n_jobs_outer=1,
#         verbose=True,
#     )

#     for roi in ROIS:
#         for predictor in UV_PREDICTORS:
#             n_subj = len(uv_results.get(roi, {}).get(predictor, {}).get('subject_list', []))
#             print(f"ROI={roi}, predictor={predictor}: {n_subj} subjects")
# else:
#     print("RUN_PROJECTION_NPZ=False; skipped")


## Single-Electrode Example (One Subject, One ROI)
Run a deterministic one-subject/one-ROI example, select one electrode, and plot electrode-level reactivation heatmaps.


In [ ]:
# =============================
# Single-electrode example: artifact preparation
# =============================

if RUN_SINGLE_ELECTRODE_EXAMPLE:
    if len(subjects) == 0:
        raise ValueError('No subjects available.')
    if SINGLE_SUBJECT not in subjects:
        raise ValueError(
            f"SINGLE_SUBJECT '{SINGLE_SUBJECT}' not found in cohort. "
            f"Example available subject: {subjects[0]}"
        )

    single_npz_path = (
        f"{trial_save_dir}/univariate/{SINGLE_ROI}/"
        f"{SINGLE_SUBJECT}_{SINGLE_PREDICTOR}_zprojection.npz"
    )

    if (not os.path.exists(single_npz_path)) and SINGLE_RUN_COMPUTE_IF_MISSING:
        print('Single example NPZ missing; running one-subject compute...')
        _single_result = run_univariate_reactivation_pipeline(
            subjects=[SINGLE_SUBJECT],
            rois=[SINGLE_ROI],
            predictors=[SINGLE_PREDICTOR],
            base_dir=base_dir,
            load_dir=load_dir,
            event_encoding=UV_EVENT_ENCODING,
            event_projection=UV_EVENT_PROJECTION,
            covariates=UV_COVARIATES,
            freq_range=UV_FREQ_RANGE,
            downsample_factor=UV_DOWNSAMPLE_FACTOR,
            win_ms=UV_WIN_MS,
            min_trials=min_trials,
            n_surrogates=N_SURROGATES,
            use_surrogates=USE_SURROGATES,
            seed=CLUSTER_SEED,
            save_trial_level=True,
            trial_save_dir=trial_save_dir,
            overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
            trial_save_dtype=np.float32,
            n_jobs_outer=1,
            verbose=True,
        )

    if not os.path.exists(single_npz_path):
        raise FileNotFoundError(
            f"Single example NPZ not found at {single_npz_path}. "
            f"Set SINGLE_RUN_COMPUTE_IF_MISSING=True or run projection pipeline first."
        )

    z_projection_single, fb_times_single, cue_times_single, meta_single = load_univariate_trial_level(
        roi=SINGLE_ROI,
        subject=SINGLE_SUBJECT,
        trial_save_dir=trial_save_dir,
        predictor=SINGLE_PREDICTOR,
    )
    memory_outcomes_single = np.asarray(meta_single['memory_outcomes'])
    print(f"Loaded single example NPZ: {single_npz_path}")
else:
    print('RUN_SINGLE_ELECTRODE_EXAMPLE=False; skipped single-electrode preparation')


In [ ]:
# =============================
# Single-electrode example: deterministic electrode selection
# =============================

if RUN_SINGLE_ELECTRODE_EXAMPLE:
    if 'electrode_labels' in meta_single:
        electrode_labels_single = np.asarray(meta_single['electrode_labels']).astype(str)
    else:
        electrode_labels_single = np.array(
            [f'elec_{i}' for i in range(z_projection_single.shape[2])],
            dtype=str
        )

    mode = SINGLE_ELECTRODE_MODE
    if mode == 'index':
        if not (0 <= int(SINGLE_ELECTRODE_INDEX) < z_projection_single.shape[2]):
            raise IndexError(
                f"SINGLE_ELECTRODE_INDEX {SINGLE_ELECTRODE_INDEX} out of bounds "
                f"for {z_projection_single.shape[2]} electrodes"
            )
        selected_elec_idx = int(SINGLE_ELECTRODE_INDEX)
    elif mode == 'label':
        if SINGLE_ELECTRODE_LABEL is None:
            raise ValueError("SINGLE_ELECTRODE_LABEL must be set when mode='label'")
        matches = np.where(electrode_labels_single == str(SINGLE_ELECTRODE_LABEL))[0]
        if len(matches) == 0:
            raise ValueError(
                f"Label '{SINGLE_ELECTRODE_LABEL}' not found. "
                f"First labels: {electrode_labels_single[:10].tolist()}"
            )
        selected_elec_idx = int(matches[0])
    elif mode == 'max_abs':
        score = np.nanmean(np.abs(z_projection_single), axis=(0, 1, 3))
        selected_elec_idx = int(np.nanargmax(score))
    else:
        raise ValueError("SINGLE_ELECTRODE_MODE must be one of: 'index', 'label', 'max_abs'")

    selected_elec_label = str(electrode_labels_single[selected_elec_idx])

    hits_mask_single = memory_outcomes_single == 'hits'
    misses_mask_single = memory_outcomes_single == 'misses'

    print('Single-electrode selection summary:')
    print(f"  subject={SINGLE_SUBJECT}, roi={SINGLE_ROI}, predictor={SINGLE_PREDICTOR}")
    print(f"  selected_electrode_idx={selected_elec_idx}, label={selected_elec_label}")
    print(f"  z_projection_shape={z_projection_single.shape}")
    print(
        f"  n_trials={z_projection_single.shape[-1]}, "
        f"n_hits={int(np.sum(hits_mask_single))}, n_misses={int(np.sum(misses_mask_single))}"
    )
else:
    print('RUN_SINGLE_ELECTRODE_EXAMPLE=False; skipped electrode selection')


In [ ]:
# =============================
# Single-electrode example: heatmap plotting + QC
# =============================

if RUN_SINGLE_ELECTRODE_EXAMPLE:
    assert len(memory_outcomes_single) == z_projection_single.shape[-1], (
        f"memory_outcomes mismatch: {len(memory_outcomes_single)} vs {z_projection_single.shape[-1]}"
    )
    assert len(electrode_labels_single) == z_projection_single.shape[2], (
        f"electrode_labels mismatch: {len(electrode_labels_single)} vs {z_projection_single.shape[2]}"
    )

    z_elec = z_projection_single[:, :, selected_elec_idx, :]  # (n_fb, n_cue, n_trials)
    all_trials_map = np.nanmean(z_elec, axis=2)

    do_hm = SINGLE_USE_HITS_MINUS_MISSES and (np.sum(hits_mask_single) > 0) and (np.sum(misses_mask_single) > 0)
    hm_map = None
    if do_hm:
        hm_map = np.nanmean(z_elec[:, :, hits_mask_single], axis=2) - np.nanmean(z_elec[:, :, misses_mask_single], axis=2)
    elif SINGLE_USE_HITS_MINUS_MISSES:
        print('Skipping hits-minus-misses panel: no hits or no misses available.')

    ncols = 2 if hm_map is not None else 1
    fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5))
    if ncols == 1:
        axes = [axes]

    vmax_a = np.nanmax(np.abs(all_trials_map))
    if not np.isfinite(vmax_a) or vmax_a == 0:
        vmax_a = 1.0

    im0 = axes[0].imshow(
        all_trials_map,
        aspect='auto',
        origin='lower',
        cmap='RdBu_r',
        vmin=-vmax_a,
        vmax=vmax_a,
        extent=[cue_times_single[0], cue_times_single[-1], fb_times_single[0], fb_times_single[-1]],
    )
    axes[0].set_title(f"All Trials Mean\n{SINGLE_SUBJECT} {SINGLE_ROI} {SINGLE_PREDICTOR} | {selected_elec_label}")
    axes[0].set_xlabel('Cue Time (s)')
    axes[0].set_ylabel('Feedback Time (s)')
    plt.colorbar(im0, ax=axes[0], label='Mean z-projection')

    if hm_map is not None:
        vmax_b = np.nanmax(np.abs(hm_map))
        if not np.isfinite(vmax_b) or vmax_b == 0:
            vmax_b = 1.0
        im1 = axes[1].imshow(
            hm_map,
            aspect='auto',
            origin='lower',
            cmap='RdBu_r',
            vmin=-vmax_b,
            vmax=vmax_b,
            extent=[cue_times_single[0], cue_times_single[-1], fb_times_single[0], fb_times_single[-1]],
        )
        axes[1].set_title(f"Hits - Misses\n{SINGLE_SUBJECT} {SINGLE_ROI} {SINGLE_PREDICTOR} | {selected_elec_label}")
        axes[1].set_xlabel('Cue Time (s)')
        axes[1].set_ylabel('Feedback Time (s)')
        plt.colorbar(im1, ax=axes[1], label='Delta z-projection')

    plt.tight_layout()
    plt.show()

    if SINGLE_SAVE_FIG:
        os.makedirs(SINGLE_FIG_DIR, exist_ok=True)
        safe_label = selected_elec_label.replace('/', '-').replace(' ', '_')
        fig_path = (
            f"{SINGLE_FIG_DIR}/"
            f"{SINGLE_SUBJECT}_{SINGLE_ROI}_{SINGLE_PREDICTOR}_{safe_label}_reactivation_heatmap.png"
        )
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure: {fig_path}")

    print('Single-electrode QC stats:')
    print(f"  min={np.nanmin(z_elec):.4f}")
    print(f"  max={np.nanmax(z_elec):.4f}")
    print(f"  mean={np.nanmean(z_elec):.4f}")
else:
    print('RUN_SINGLE_ELECTRODE_EXAMPLE=False; skipped plotting/QC')


In [ ]:
# =============================
# Load + summarize saved projections
# =============================

summary_rows = []
subject_diff_maps = {roi: {pred: [] for pred in UV_PREDICTORS} for roi in ROIS}

for roi in ROIS:
    for predictor in UV_PREDICTORS:
        n_loaded = 0
        for subj in subjects:
            try:
                z_proj, fb_times, cue_times, meta = load_univariate_trial_level(
                    roi=roi,
                    subject=subj,
                    trial_save_dir=trial_save_dir,
                    predictor=predictor,
                )
                memory_outcomes = np.asarray(meta['memory_outcomes'])
                s = summarize_univariate_hits_misses(z_proj, memory_outcomes)
                subject_diff_maps[roi][predictor].append(s['diff_agg'])
                n_loaded += 1
                summary_rows.append({
                    'subject': subj,
                    'roi': roi,
                    'predictor': predictor,
                    'n_trials': int(z_proj.shape[-1]),
                    'n_electrodes': int(z_proj.shape[2]),
                    'n_hits': int(np.sum(s['hits_mask'])),
                    'n_misses': int(np.sum(s['misses_mask'])),
                })
            except Exception:
                continue
        print(f"Loaded summaries: ROI={roi}, predictor={predictor}, n_subjects={n_loaded}")

summary_df = pd.DataFrame(summary_rows)
print(summary_df.head())


In [ ]:
# =============================
# Group-level plots + stats
# =============================

if RUN_GROUP_STATS:
    # Mean hits-minus-misses heatmap per ROI/predictor
    for roi in ROIS:
        for predictor in UV_PREDICTORS:
            maps = subject_diff_maps[roi][predictor]
            if len(maps) == 0:
                continue
            mean_map = np.mean(np.stack(maps, axis=0), axis=0)
            vmax = np.nanmax(np.abs(mean_map))
            if not np.isfinite(vmax) or vmax == 0:
                vmax = 1.0

            plt.figure(figsize=(5, 4))
            plt.imshow(mean_map, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            plt.colorbar(label='Hits - Misses (mean z-projection)')
            plt.title(f'{roi} / {predictor} (n={len(maps)} subjects)')
            plt.xlabel('Cue Time Index')
            plt.ylabel('Feedback Time Index')
            plt.tight_layout()
            plt.show()

    # Optional binomial test: proportion of significant electrodes > 0.05
    binom_rows = []
    for roi in ROIS:
        for predictor in UV_PREDICTORS:
            n_total = 0
            n_sig = 0
            for subj in subjects:
                try:
                    z_proj, _, _, meta = load_univariate_trial_level(
                        roi=roi,
                        subject=subj,
                        trial_save_dir=trial_save_dir,
                        predictor=predictor,
                    )
                    memory_outcomes = np.asarray(meta['memory_outcomes'])
                    counts = run_per_electrode_2sample_cluster_test(
                        z_reactivation=z_proj,
                        memory_outcomes=memory_outcomes,
                        n_permutations=N_PERMUTATIONS_CLUSTER,
                        cluster_seed=CLUSTER_SEED,
                        verbose=False,
                    )
                    n_total += len(counts)
                    n_sig += int(np.sum(np.asarray(counts) > 0))
                except Exception:
                    continue

            if n_total > 0:
                pval = binomtest(n_sig, n_total, p=0.05, alternative='greater').pvalue
                prop = n_sig / n_total
            else:
                pval = np.nan
                prop = np.nan
            binom_rows.append({
                'roi': roi,
                'predictor': predictor,
                'n_sig': n_sig,
                'n_total': n_total,
                'proportion': prop,
                'pvalue': pval,
            })

    binom_df = pd.DataFrame(binom_rows)
    print("\nBinomial test summary (H1: proportion > 0.05):")
    print(binom_df.to_string(index=False))
else:
    print("RUN_GROUP_STATS=False; skipped")


In [ ]:
# =============================
# Plot all significant electrodes for an input ROI
# =============================

SIG_PLOT_ROI = 'dlPFC'            # Input ROI
SIG_PLOT_PREDICTOR = UV_PREDICTORS[0]
SIG_PLOT_MAX_PER_FIG = 12
SIG_PLOT_N_COLS = 4

sig_entries = []

for subj in subjects:
    try:
        z_proj, fb_times_sig, cue_times_sig, meta_sig = load_univariate_trial_level(
            roi=SIG_PLOT_ROI,
            subject=subj,
            trial_save_dir=trial_save_dir,
            predictor=SIG_PLOT_PREDICTOR,
        )
        memory_outcomes_sig = np.asarray(meta_sig['memory_outcomes'])

        # Need at least 2 trials per group for stable permutation testing.
        if np.sum(memory_outcomes_sig == 'hits') < 2 or np.sum(memory_outcomes_sig == 'misses') < 2:
            continue

        cluster_counts = run_per_electrode_2sample_cluster_test(
            z_reactivation=z_proj,
            memory_outcomes=memory_outcomes_sig,
            n_permutations=N_PERMUTATIONS_CLUSTER,
            cluster_seed=CLUSTER_SEED,
            verbose=False,
        )

        summary_sig = summarize_univariate_hits_misses(z_proj, memory_outcomes_sig)
        diff_per_electrode = summary_sig['diff_per_electrode']  # (n_fb, n_cue, n_elec)

        if 'electrode_labels' in meta_sig:
            elec_labels_sig = np.asarray(meta_sig['electrode_labels']).astype(str)
        else:
            elec_labels_sig = np.array([f'elec_{i}' for i in range(z_proj.shape[2])], dtype=str)

        sig_electrode_idx = np.where(np.asarray(cluster_counts) > 0)[0]
        for e_idx in sig_electrode_idx:
            sig_entries.append({
                'subject': subj,
                'electrode_idx': int(e_idx),
                'electrode_label': str(elec_labels_sig[e_idx]),
                'n_sig_clusters': int(cluster_counts[e_idx]),
                'map': diff_per_electrode[:, :, e_idx],
                'fb_times': fb_times_sig,
                'cue_times': cue_times_sig,
            })
    except Exception:
        continue

print(f"ROI={SIG_PLOT_ROI}, predictor={SIG_PLOT_PREDICTOR}: found {len(sig_entries)} significant electrodes across subjects")

if len(sig_entries) == 0:
    print('No significant electrodes found for this ROI/predictor.')
else:
    all_maps = [entry['map'] for entry in sig_entries]
    vmax_global = np.nanmax([np.nanmax(np.abs(m)) for m in all_maps if np.any(np.isfinite(m))])
    if not np.isfinite(vmax_global) or vmax_global == 0:
        vmax_global = 1.0

    for page_start in range(0, len(sig_entries), SIG_PLOT_MAX_PER_FIG):
        page = sig_entries[page_start:page_start + SIG_PLOT_MAX_PER_FIG]
        n_panels = len(page)
        n_cols = SIG_PLOT_N_COLS
        n_rows = int(np.ceil(n_panels / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = np.array([[axes]])
        elif n_rows == 1:
            axes = axes.reshape(1, -1)

        for idx, entry in enumerate(page):
            r, c = idx // n_cols, idx % n_cols
            ax = axes[r, c]
            m = entry['map']
            im = ax.imshow(
                m,
                aspect='auto',
                origin='lower',
                cmap='RdBu_r',
                vmin=-vmax_global,
                vmax=vmax_global,
                extent=[entry['cue_times'][0], entry['cue_times'][-1], entry['fb_times'][0], entry['fb_times'][-1]],
            )
            ax.set_title(
                f"{entry['subject']} | {entry['electrode_label']}\n"
                f"idx={entry['electrode_idx']} | n_sig={entry['n_sig_clusters']}",
                fontsize=10,
            )
            ax.set_xlabel('Cue Time (s)')
            ax.set_ylabel('Feedback Time (s)')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        for idx in range(n_panels, n_rows * n_cols):
            r, c = idx // n_cols, idx % n_cols
            axes[r, c].set_visible(False)

        fig.suptitle(
            f"Significant Electrodes | ROI={SIG_PLOT_ROI} | predictor={SIG_PLOT_PREDICTOR} | "
            f"{page_start + 1}-{page_start + n_panels} of {len(sig_entries)}",
            fontsize=12,
            y=1.02,
        )
        plt.tight_layout()
        plt.show()


In [ ]:
# =============================
# Output index
# =============================

print("\nGenerated / expected outputs:")

if len(encoding_csv_paths) > 0:
    print("\nLegacy-compatible encoding CSVs:")
    for predictor, path in encoding_csv_paths.items():
        print(f"  {predictor}: {path}")

print("\nUnivariate NPZ projection files (pattern):")
for roi in ROIS:
    for predictor in UV_PREDICTORS:
        print(f"  {trial_save_dir}/univariate/{roi}/<SUBJECT>_{predictor}_zprojection.npz")

if 'summary_df' in locals() and len(summary_df) > 0:
    print("\nProjection summary rows:", len(summary_df))
